[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/mathematical_reasoning/computation.ipynb)

# Computation: Mathematical Reasoning

Interactive demonstrations of mathematical reasoning concepts using Python and SymPy.

**Topics covered:**
1. Symbolic logic with SymPy
2. Induction verification with numerical examples
3. Set operations demonstration
4. Problem-solving worked examples (Fermi estimation, dimensional checking)

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import sympy as sp
from sympy.logic.boolalg import truth_table, And, Or, Not, Implies, Equivalent
from sympy import symbols, simplify, Rational, oo, Sum, factorial
from itertools import product as iter_product

rng = np.random.default_rng(seed=42)

# For nicer output
sp.init_printing(use_unicode=True)

---
## 1. Symbolic Logic with SymPy

We use SymPy's logic module to construct truth tables, verify logical equivalences,
and explore the structure of propositional logic.

### 1.1 Truth Tables

In [ ]:
# Define propositional variables
p, q, r = symbols('p q r')


def print_truth_table(expr, variables, name=""):
    """
    Print truth table for a logical expression.
    """
    var_names = [str(v) for v in variables]
    header = "  ".join(f"{v:>5}" for v in var_names) + f"  | {name if name else str(expr):>20}"
    print(header)
    print("-" * len(header))

    for values in iter_product([True, False], repeat=len(variables)):
        subs = dict(zip(variables, values))
        result = bool(expr.subs(subs))
        row = "  ".join(f"{str(v):>5}" for v in values)
        print(f"{row}  | {str(result):>20}")
    print()


# Basic connectives
print("=" * 50)
print("IMPLICATION: p → q")
print("=" * 50)
print_truth_table(Implies(p, q), [p, q], "p → q")

print("Key insight: p → q is FALSE only when p=True and q=False")
print("When the hypothesis is false, the implication is vacuously true.")

In [ ]:
# Compare implication and its contrapositive
implication = Implies(p, q)
contrapositive = Implies(Not(q), Not(p))

print("=" * 50)
print("CONTRAPOSITIVE EQUIVALENCE")
print("=" * 50)
print("\nImplication: p → q")
print_truth_table(implication, [p, q], "p → q")

print("Contrapositive: ¬q → ¬p")
print_truth_table(contrapositive, [p, q], "¬q → ¬p")

# Verify equivalence symbolically
equiv = Equivalent(implication, contrapositive)
print(f"Are they logically equivalent? {bool(simplify(equiv))}")
print("\nThis is why proof by contrapositive works!")

### 1.2 De Morgan's Laws

In [ ]:
# De Morgan's Laws verification
print("=" * 50)
print("DE MORGAN'S LAWS")
print("=" * 50)

# Law 1: ¬(p ∧ q) ≡ ¬p ∨ ¬q
lhs1 = Not(And(p, q))
rhs1 = Or(Not(p), Not(q))

print("\nLaw 1: ¬(p ∧ q) ≡ ¬p ∨ ¬q")
print_truth_table(lhs1, [p, q], "¬(p ∧ q)")
print_truth_table(rhs1, [p, q], "¬p ∨ ¬q")
print(f"Equivalent: {bool(simplify(Equivalent(lhs1, rhs1)))}")

# Law 2: ¬(p ∨ q) ≡ ¬p ∧ ¬q
lhs2 = Not(Or(p, q))
rhs2 = And(Not(p), Not(q))

print(f"\nLaw 2: ¬(p ∨ q) ≡ ¬p ∧ ¬q")
print(f"Equivalent: {bool(simplify(Equivalent(lhs2, rhs2)))}")

print("\nModeling connection:")
print('"NOT (assumption 1 AND assumption 2)" ≡ "NOT assumption 1 OR NOT assumption 2"')
print("When a model fails, at least one assumption must be violated.")

### 1.3 Logical Equivalences — Comprehensive Verification

In [ ]:
# Verify multiple logical equivalences
equivalences = [
    ("Double negation",       Not(Not(p)),              p),
    ("Contrapositive",        Implies(p, q),            Implies(Not(q), Not(p))),
    ("Implication as ∨",      Implies(p, q),            Or(Not(p), q)),
    ("¬(p → q)",              Not(Implies(p, q)),       And(p, Not(q))),
    ("De Morgan 1",           Not(And(p, q)),           Or(Not(p), Not(q))),
    ("De Morgan 2",           Not(Or(p, q)),            And(Not(p), Not(q))),
    ("Distributivity",        And(p, Or(q, r)),         Or(And(p, q), And(p, r))),
    ("Absorption",            Or(p, And(p, q)),         p),
]

print(f"{'Name':<25} {'LHS':<30} {'RHS':<30} {'Equiv?'}")
print("=" * 95)
for name, lhs, rhs in equivalences:
    is_equiv = bool(simplify(Equivalent(lhs, rhs)))
    mark = "✓" if is_equiv else "✗"
    print(f"{name:<25} {str(lhs):<30} {str(rhs):<30} {mark}")

---
## 2. Induction Verification with Numerical Examples

While numerical checks do not constitute a proof, they help build confidence
and catch errors in proposed formulas before attempting a formal proof.

### 2.1 Sum of First $n$ Natural Numbers

In [ ]:
def verify_sum_formula(n_max=20):
    """
    Verify: 1 + 2 + ... + n = n(n+1)/2
    """
    print(f"{'n':>4}  {'Σi (direct)':>15}  {'n(n+1)/2':>15}  {'Match':>6}")
    print("-" * 48)

    all_match = True
    for n in range(1, n_max + 1):
        direct_sum = sum(range(1, n + 1))
        formula = n * (n + 1) // 2
        match = direct_sum == formula
        all_match &= match
        if n <= 10 or n == n_max:
            print(f"{n:>4}  {direct_sum:>15}  {formula:>15}  {'✓' if match else '✗':>6}")
        elif n == 11:
            print(f"{'...':>4}  {'...':>15}  {'...':>15}  {'...':>6}")

    print(f"\nAll values match for n = 1 to {n_max}: {all_match}")
    print("(Proved rigorously by induction in theory.md, Section 5.2)")


verify_sum_formula()

### 2.2 Geometric Series

In [ ]:
def verify_geometric_series(r_val, n_max=15):
    """
    Verify: Σ_{i=0}^{n} r^i = (r^{n+1} - 1) / (r - 1) for r ≠ 1
    """
    print(f"Geometric series with r = {r_val}:")
    print(f"{'n':>4}  {'Σr^i (direct)':>20}  {'Formula':>20}  {'Error':>12}")
    print("-" * 62)

    max_error = 0
    for n in range(0, n_max + 1):
        direct = sum(r_val**i for i in range(n + 1))
        formula = (r_val**(n + 1) - 1) / (r_val - 1)
        error = abs(direct - formula)
        max_error = max(max_error, error)
        print(f"{n:>4}  {direct:>20.6f}  {formula:>20.6f}  {error:>12.2e}")

    print(f"\nMaximum numerical error: {max_error:.2e}")
    print(f"(Machine epsilon ≈ {np.finfo(float).eps:.2e})")
    print()


verify_geometric_series(r_val=0.5)
verify_geometric_series(r_val=2.0, n_max=10)

### 2.3 Sum of Squares

In [ ]:
def verify_sum_of_squares(n_max=15):
    """
    Verify: 1² + 2² + ... + n² = n(n+1)(2n+1)/6
    """
    print("Sum of squares formula: Σi² = n(n+1)(2n+1)/6")
    print(f"{'n':>4}  {'Σi² (direct)':>15}  {'Formula':>15}  {'Match':>6}")
    print("-" * 48)

    all_match = True
    for n in range(1, n_max + 1):
        direct = sum(i**2 for i in range(1, n + 1))
        formula = n * (n + 1) * (2 * n + 1) // 6
        match = direct == formula
        all_match &= match
        print(f"{n:>4}  {direct:>15}  {formula:>15}  {'✓' if match else '✗':>6}")

    print(f"\nAll match: {all_match}")


verify_sum_of_squares()

### 2.4 Symbolic Induction with SymPy

In [ ]:
# Use SymPy to verify summation identities symbolically
n, k, i = symbols('n k i', positive=True, integer=True)
r_sym = symbols('r')

# Sum of first n integers
sum_formula = sp.summation(i, (i, 1, n))
print(f"SymPy computes Σ(i, 1..n) = {sum_formula}")
print(f"Simplified: {sp.simplify(sum_formula)}")
print(f"Expected: n(n+1)/2 = {n * (n + 1) / 2}")
print(f"Match: {sp.simplify(sum_formula - n * (n + 1) / 2) == 0}")

print()

# Sum of squares
sum_sq = sp.summation(i**2, (i, 1, n))
print(f"Σ(i², 1..n) = {sum_sq}")
print(f"Expected: n(n+1)(2n+1)/6")
print(f"Match: {sp.simplify(sum_sq - n * (n + 1) * (2 * n + 1) / 6) == 0}")

print()

# Sum of cubes
sum_cubes = sp.summation(i**3, (i, 1, n))
print(f"Σ(i³, 1..n) = {sum_cubes}")
print(f"This equals [n(n+1)/2]² — the sum of cubes is the square of the sum!")
print(f"Match: {sp.simplify(sum_cubes - (n * (n + 1) / 2)**2) == 0}")

### 2.5 Inductive Step Verification

We can also verify the inductive step symbolically: assuming $P(k)$, does $P(k+1)$ follow?

In [ ]:
# Verify inductive step for sum formula
# Claim: if Σ(i, 1..k) = k(k+1)/2, then Σ(i, 1..k+1) = (k+1)(k+2)/2

# Inductive hypothesis: Σ(i, 1..k) = k(k+1)/2
IH = k * (k + 1) / 2

# LHS of P(k+1): Σ(i, 1..k+1) = Σ(i, 1..k) + (k+1) = IH + (k+1)
lhs_step = IH + (k + 1)

# RHS of P(k+1): (k+1)(k+2)/2
rhs_step = (k + 1) * (k + 2) / 2

print("Inductive step verification for Σi = n(n+1)/2:\n")
print(f"  Inductive hypothesis: Σ(i, 1..k) = {IH}")
print(f"  LHS of P(k+1): Σ(i, 1..k) + (k+1) = {IH} + (k+1) = {sp.expand(lhs_step)}")
print(f"  RHS of P(k+1): (k+1)(k+2)/2 = {sp.expand(rhs_step)}")
print(f"  LHS - RHS = {sp.simplify(lhs_step - rhs_step)}")
print(f"  Inductive step holds: {sp.simplify(lhs_step - rhs_step) == 0} ✓")

---
## 3. Set Operations Demonstration

Using Python's built-in set type to demonstrate set operations and verify identities.

In [ ]:
# Define some sets
A = {1, 2, 3, 4, 5}
B = {3, 4, 5, 6, 7}
C = {5, 6, 7, 8, 9}
U = set(range(1, 11))  # Universal set {1, 2, ..., 10}

print(f"A = {sorted(A)}")
print(f"B = {sorted(B)}")
print(f"C = {sorted(C)}")
print(f"U = {sorted(U)}")

print(f"\n--- Basic Operations ---")
print(f"A ∪ B         = {sorted(A | B)}")
print(f"A ∩ B         = {sorted(A & B)}")
print(f"A \ B         = {sorted(A - B)}")
print(f"B \ A         = {sorted(B - A)}")
print(f"A △ B         = {sorted(A ^ B)}")
print(f"A^c (in U)    = {sorted(U - A)}")
print(f"|A| = {len(A)}, |B| = {len(B)}, |A ∩ B| = {len(A & B)}")

In [ ]:
# Verify key identities
print("=" * 50)
print("VERIFYING SET IDENTITIES")
print("=" * 50)

# Inclusion-Exclusion
ie_lhs = len(A | B)
ie_rhs = len(A) + len(B) - len(A & B)
print(f"\nInclusion-Exclusion: |A ∪ B| = |A| + |B| - |A ∩ B|")
print(f"  LHS: |A ∪ B| = {ie_lhs}")
print(f"  RHS: {len(A)} + {len(B)} - {len(A & B)} = {ie_rhs}")
print(f"  ✓" if ie_lhs == ie_rhs else "  ✗")

# De Morgan's Laws for sets
dm1_lhs = U - (A | B)          # (A ∪ B)^c
dm1_rhs = (U - A) & (U - B)    # A^c ∩ B^c
print(f"\nDe Morgan 1: (A ∪ B)^c = A^c ∩ B^c")
print(f"  LHS: {sorted(dm1_lhs)}")
print(f"  RHS: {sorted(dm1_rhs)}")
print(f"  Equal: {dm1_lhs == dm1_rhs} ✓")

dm2_lhs = U - (A & B)          # (A ∩ B)^c
dm2_rhs = (U - A) | (U - B)    # A^c ∪ B^c
print(f"\nDe Morgan 2: (A ∩ B)^c = A^c ∪ B^c")
print(f"  LHS: {sorted(dm2_lhs)}")
print(f"  RHS: {sorted(dm2_rhs)}")
print(f"  Equal: {dm2_lhs == dm2_rhs} ✓")

# Distributivity
dist_lhs = A & (B | C)
dist_rhs = (A & B) | (A & C)
print(f"\nDistributivity: A ∩ (B ∪ C) = (A ∩ B) ∪ (A ∩ C)")
print(f"  LHS: {sorted(dist_lhs)}")
print(f"  RHS: {sorted(dist_rhs)}")
print(f"  Equal: {dist_lhs == dist_rhs} ✓")

In [ ]:
# Power set and cardinality
S = {1, 2, 3}
power_set = []
for r_val in range(len(S) + 1):
    from itertools import combinations
    for subset in combinations(sorted(S), r_val):
        power_set.append(set(subset))

print(f"Set S = {sorted(S)}")
print(f"Power set P(S) has {len(power_set)} elements (= 2^{len(S)} = {2**len(S)}):")
for i, subset in enumerate(power_set):
    print(f"  {sorted(subset) if subset else '∅'}")

### 3.1 Cartesian Product

In [ ]:
# Cartesian product
X = {1, 2, 3}
Y = {'a', 'b'}

cart_product = {(x, y) for x in sorted(X) for y in sorted(Y)}

print(f"X = {sorted(X)}")
print(f"Y = {sorted(Y)}")
print(f"X × Y = {sorted(cart_product)}")
print(f"|X × Y| = {len(cart_product)} = |X| × |Y| = {len(X)} × {len(Y)} = {len(X) * len(Y)} ✓")

print("\nModeling connection:")
print("If X = parameter space and Y = initial conditions,")
print("then X × Y = space of all model configurations.")

### 3.2 Relations: Checking Properties

In [ ]:
def check_relation_properties(S, R, name="R"):
    """
    Check if a binary relation R on set S is reflexive, symmetric,
    antisymmetric, and transitive.
    """
    print(f"Relation {name} on S = {sorted(S)}:")
    print(f"  R = {sorted(R)}\n")

    # Reflexive: ∀a ∈ S, (a,a) ∈ R
    reflexive = all((a, a) in R for a in S)
    print(f"  Reflexive:     {reflexive}")

    # Symmetric: (a,b) ∈ R → (b,a) ∈ R
    symmetric = all((b, a) in R for (a, b) in R)
    print(f"  Symmetric:     {symmetric}")

    # Antisymmetric: (a,b) ∈ R ∧ (b,a) ∈ R → a = b
    antisymmetric = all(a == b for (a, b) in R if (b, a) in R)
    print(f"  Antisymmetric: {antisymmetric}")

    # Transitive: (a,b) ∈ R ∧ (b,c) ∈ R → (a,c) ∈ R
    transitive = all((a, c) in R
                     for (a, b1) in R
                     for (b2, c) in R
                     if b1 == b2)
    print(f"  Transitive:    {transitive}")

    # Classifications
    if reflexive and symmetric and transitive:
        print(f"  → {name} is an EQUIVALENCE RELATION")
    if reflexive and antisymmetric and transitive:
        print(f"  → {name} is a PARTIAL ORDER")
    print()


S_rel = {1, 2, 3, 4}

# Example 1: Divisibility relation
divides = {(a, b) for a in S_rel for b in S_rel if b % a == 0}
check_relation_properties(S_rel, divides, "divides")

# Example 2: "same parity" relation (equivalence relation)
same_parity = {(a, b) for a in S_rel for b in S_rel if a % 2 == b % 2}
check_relation_properties(S_rel, same_parity, "same_parity")

### 3.3 Functions: Injectivity, Surjectivity, Bijectivity

In [ ]:
def check_function_properties(f, domain, codomain, name="f"):
    """
    Check if f: domain → codomain is injective, surjective, bijective.
    f is given as a dictionary.
    """
    image = set(f.values())

    # Well-defined: every domain element maps to codomain
    well_defined = all(f[x] in codomain for x in domain)

    # Injective: f(a) = f(b) → a = b
    values = list(f.values())
    injective = len(values) == len(set(values))

    # Surjective: every codomain element is hit
    surjective = image == codomain

    bijective = injective and surjective

    print(f"Function {name}: {sorted(domain)} → {sorted(codomain)}")
    print(f"  Mapping: {dict(sorted(f.items()))}")
    print(f"  Image:   {sorted(image)}")
    print(f"  Well-defined: {well_defined}")
    print(f"  Injective (one-to-one): {injective}")
    print(f"  Surjective (onto):      {surjective}")
    print(f"  Bijective:              {bijective}")
    print()


dom = {1, 2, 3}
cod = {'a', 'b', 'c'}

# Bijective function
f1 = {1: 'a', 2: 'b', 3: 'c'}
check_function_properties(f1, dom, cod, "f₁ (bijective)")

# Not injective (2 and 3 map to same value)
cod2 = {'a', 'b', 'c'}
f2 = {1: 'a', 2: 'b', 3: 'b'}
check_function_properties(f2, dom, cod2, "f₂ (not injective)")

# Injective but not surjective
cod3 = {'a', 'b', 'c', 'd'}
f3 = {1: 'a', 2: 'c', 3: 'd'}
check_function_properties(f3, dom, cod3, "f₃ (injective, not surjective)")

---
## 4. Problem-Solving Worked Examples

### 4.1 Fermi Estimation

In [ ]:
def fermi_estimate(question, steps, final_calc, actual=None):
    """
    Walk through a Fermi estimation problem.

    Parameters:
        question: str — the question
        steps: list of (description, value) — estimation steps
        final_calc: callable — function of step values → estimate
        actual: float or None — actual value for comparison
    """
    print(f"QUESTION: {question}")
    print("=" * 60)

    values = []
    for i, (desc, val) in enumerate(steps, 1):
        print(f"  Step {i}: {desc}")
        print(f"          Estimate: {val:,.0f}" if isinstance(val, (int, float)) else f"          Estimate: {val}")
        values.append(val)

    result = final_calc(*values)
    print(f"\n  ESTIMATE: {result:,.0f}")

    if actual is not None:
        ratio = result / actual
        print(f"  ACTUAL:   {actual:,.0f}")
        print(f"  Ratio:    {ratio:.2f}x")
        print(f"  Order of magnitude: {'✓ CORRECT' if 0.1 <= ratio <= 10 else '✗ OFF'}")
    print()


# Example 1: How many piano tuners in Chicago?
fermi_estimate(
    "How many piano tuners are in Chicago?",
    [
        ("Chicago population", 3_000_000),
        ("People per household", 2.5),
        ("Fraction of households with piano (~5%)", 0.05),
        ("Tunings per piano per year", 1),
        ("Tunings per tuner per day", 4),
        ("Working days per year", 250),
    ],
    lambda pop, pph, frac, tpy, tpd, wdy:
        (pop / pph) * frac * tpy / (tpd * wdy),
    actual=100
)

# Example 2: How many golf balls fit in a school bus?
fermi_estimate(
    "How many golf balls fit in a school bus?",
    [
        ("Bus interior length (m)", 7.5),
        ("Bus interior width (m)", 2.0),
        ("Bus interior height (m)", 1.8),
        ("Bus interior volume (m³)", 27),
        ("Golf ball diameter (cm) → radius ≈ 2.1cm", 0.021),
        ("Packing efficiency (random ~64%)", 0.64),
    ],
    lambda l, w, h, vol, r, eff:
        vol * eff / ((4/3) * np.pi * r**3),
    actual=500_000
)

### 4.2 Dimensional Analysis / Checking

In [ ]:
class Dimension:
    """
    Simple dimension tracking for dimensional analysis.
    Represents dimensions as (length, mass, time) exponents.
    """
    def __init__(self, name, L=0, M=0, T=0):
        self.name = name
        self.L = L  # length
        self.M = M  # mass
        self.T = T  # time

    def __repr__(self):
        parts = []
        if self.L: parts.append(f"L^{self.L}" if self.L != 1 else "L")
        if self.M: parts.append(f"M^{self.M}" if self.M != 1 else "M")
        if self.T: parts.append(f"T^{self.T}" if self.T != 1 else "T")
        dim_str = " · ".join(parts) if parts else "dimensionless"
        return f"{self.name}: [{dim_str}]"

    def __mul__(self, other):
        return Dimension(
            f"{self.name}·{other.name}",
            self.L + other.L,
            self.M + other.M,
            self.T + other.T
        )

    def __truediv__(self, other):
        return Dimension(
            f"{self.name}/{other.name}",
            self.L - other.L,
            self.M - other.M,
            self.T - other.T
        )

    def matches(self, other):
        return self.L == other.L and self.M == other.M and self.T == other.T


# Define base dimensions
length = Dimension("length", L=1)
mass = Dimension("mass", M=1)
time = Dimension("time", T=1)

# Derived dimensions
velocity = length / time
acceleration = velocity / time
force = mass * acceleration
energy = force * length

print("Base and derived dimensions:")
print(f"  {length}")
print(f"  {mass}")
print(f"  {time}")
print(f"  {velocity}")
print(f"  {acceleration}")
print(f"  {force}")
print(f"  {energy}")

In [ ]:
# Dimensional check: v = v₀ + a·t
print("\nDimensional check: v = v₀ + a·t")
print("=" * 40)

v = Dimension("v", L=1, T=-1)
v0 = Dimension("v₀", L=1, T=-1)
a = Dimension("a", L=1, T=-2)
t = Dimension("t", T=1)

at = a * t  # acceleration × time

print(f"  [v]    = {v}")
print(f"  [v₀]   = {v0}")
print(f"  [a·t]  = {at}")
print(f"  [v] matches [v₀]: {v.matches(v0)} ✓")
print(f"  [v] matches [a·t]: {v.matches(at)} ✓")
print(f"  All terms have same dimension: ✓")

In [ ]:
# Dimensional check for logistic growth: dP/dt = rP(1 - P/K)
print("\nDimensional check: dP/dt = rP(1 - P/K)")
print("=" * 50)
print("\nLet [P] = individuals, [t] = time\n")

# We use a generic dimension system
# [P] is just a count, not L, M, or T — we'll track symbolically

quantities = {
    'dP/dt': 'individuals / time',
    'r':     '1 / time',
    'P':     'individuals',
    'K':     'individuals',
    'P/K':   'dimensionless',
    '1-P/K': 'dimensionless',
    'rP':    'individuals / time',
    'rP(1-P/K)': 'individuals / time',
}

print(f"{'Quantity':<15} {'Dimension':<25}")
print("-" * 40)
for qty, dim in quantities.items():
    print(f"{qty:<15} [{dim}]")

print(f"\n[dP/dt] = [{quantities['dP/dt']}]")
print(f"[rP(1-P/K)] = [{quantities['rP(1-P/K)']}]")
print(f"Dimensions match: ✓")
print("\nIf r were given in year⁻¹ but t in days,")
print("the equation would be dimensionally INCONSISTENT → error!")

### 4.3 Limiting Cases Check

In [ ]:
def check_limiting_cases(formula_name, formula, variable, cases, params=None):
    """
    Check a formula against known limiting cases.

    Parameters:
        formula_name: str
        formula: sympy expression
        variable: sympy symbol to substitute
        cases: list of (value, expected_result, description)
        params: dict of parameter substitutions
    """
    print(f"Limiting cases for: {formula_name}")
    print(f"Formula: {formula}")
    print("=" * 60)

    for val, expected, desc in cases:
        subs_dict = {variable: val}
        if params:
            subs_dict.update(params)

        result = sp.simplify(formula.subs(subs_dict))

        if expected is not None:
            match = sp.simplify(result - expected) == 0
            mark = "✓" if match else "✗"
        else:
            match = None
            mark = "—"

        print(f"  {variable}={val}: {formula_name} = {result}"
              f"  (expected: {expected})  {mark}  [{desc}]")
    print()


n_sym = sp.Symbol('n', positive=True, integer=True)

# Sum formula
sum_expr = n_sym * (n_sym + 1) / 2
check_limiting_cases(
    "S(n) = n(n+1)/2",
    sum_expr,
    n_sym,
    [
        (1, 1, "just the number 1"),
        (2, 3, "1 + 2 = 3"),
        (10, 55, "well-known value"),
        (100, 5050, "Gauss's famous result"),
    ]
)

# Binomial coefficient
k_sym = sp.Symbol('k', positive=True, integer=True)
binom = sp.binomial(n_sym, k_sym)

check_limiting_cases(
    "C(n,k)",
    binom,
    k_sym,
    [
        (0, 1, "choosing nothing"),
        (1, n_sym, "choosing one element"),
        (n_sym, 1, "choosing everything"),
    ]
)

### 4.4 Polya's Method — Worked Example

We walk through a problem using Polya's four-step method.

In [ ]:
print("""
════════════════════════════════════════════════════════════════
PROBLEM: How many handshakes occur if n people each shake
         hands with every other person exactly once?
════════════════════════════════════════════════════════════════

STEP 1: UNDERSTAND THE PROBLEM
─────────────────────────────────────────────────
  • Unknown: Total number of handshakes H(n)
  • Data: n people
  • Conditions: Each pair shakes hands exactly once
  • Key insight: Order doesn't matter (A shaking B = B shaking A)

STEP 2: DEVISE A PLAN
─────────────────────────────────────────────────
  Strategy 1: Try small cases and look for a pattern
  Strategy 2: Each person shakes n-1 hands, but we double-count
  Strategy 3: This is choosing 2 people from n → C(n, 2)
""")

# Step 3: Carry out the plan
print("STEP 3: CARRY OUT THE PLAN")
print("─" * 50)
print("\n  Strategy 1 — Small cases:")
for n_val in range(2, 8):
    # Direct count: C(n, 2)
    h = n_val * (n_val - 1) // 2
    print(f"    n={n_val}: H = {h}")

print("\n  Pattern: H(n) = n(n-1)/2")
print("\n  Strategy 2 — Double counting:")
print("    Each of n people shakes n-1 hands = n(n-1) total")
print("    Each handshake counted twice → H = n(n-1)/2")
print("\n  Strategy 3 — Combinatorial:")
print("    H = C(n, 2) = n! / (2!(n-2)!) = n(n-1)/2")
print("\n  All three strategies give the same answer. ✓")

# Step 4: Look back
print("\nSTEP 4: LOOK BACK")
print("─" * 50)
n_sym = sp.Symbol('n')
H = n_sym * (n_sym - 1) / 2
print(f"  Formula: H(n) = {H}")
print(f"  Check H(2) = {H.subs(n_sym, 2)} (1 handshake) ✓")
print(f"  Check H(3) = {H.subs(n_sym, 3)} (triangle) ✓")
print(f"  Check H(1) = {H.subs(n_sym, 1)} (no one to shake with) ✓")
print(f"  Generalization: This is identical to counting edges in K_n!")
print(f"  Connection to graph theory: |E(K_n)| = C(n,2) = n(n-1)/2")

---
## Summary

In this notebook we demonstrated:

| Topic | What We Did |
|-------|-------------|
| **Propositional logic** | Built truth tables, verified contrapositive, De Morgan's laws, and other equivalences |
| **Induction (numerical)** | Checked sum, geometric series, and sum-of-squares formulas for many values |
| **Induction (symbolic)** | Used SymPy to verify summation identities and the inductive step |
| **Set operations** | Demonstrated union, intersection, difference, complement; verified identities |
| **Relations** | Checked reflexivity, symmetry, antisymmetry, transitivity |
| **Functions** | Tested injectivity, surjectivity, bijectivity |
| **Fermi estimation** | Piano tuners, golf balls — order-of-magnitude reasoning |
| **Dimensional analysis** | Built a dimension tracker, checked physics and modeling equations |
| **Limiting cases** | Verified formulas against known special values |
| **Polya's method** | Worked through the handshake problem step by step |